In [1]:
import pandas as pd

# Le chemin exact vers notre fichier CSV
chemin_csv = '/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/clinical_data(labels).csv'

# Lecture du fichier
df_clinique = pd.read_csv(chemin_csv)

# On affiche la taille du tableau (patients, variables)
print(f"Dimensions du dataset clinique : {df_clinique.shape}")

# On affiche les 5 premières lignes pour voir le contenu
df_clinique.head()

Dimensions du dataset clinique : (1063, 26)


,bcr_patient_barcode,Time,age_at_initial_pathologic_diagnosis,lymph_node_examined_count,vital_status,tissue_prospective_collection_indicator_YES,radiation_therapy_NO,breast_carcinoma_surgical_procedure_name_Lumpectomy,breast_carcinoma_surgical_procedure_name_Other,breast_carcinoma_surgical_procedure_name_Simple Mastectomy,...,pathologic_N_N1,pathologic_N_N2,pathologic_N_N3,pathologic_N_NX,pathologic_M_M1,pathologic_M_MX,pathologic_stage_Stage I,pathologic_stage_Stage III,pathologic_stage_Stage IV,pathologic_stage_Stage X
0,TCGA-E2-A1BD,1133.0,53,1.0,1,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,TCGA-BH-A0AW,622.0,56,12.0,1,False,False,True,False,False,...,True,False,False,False,False,False,False,False,False,False
2,TCGA-AO-A0JB,1542.0,50,14.0,1,False,False,False,False,False,...,True,False,False,False,False,False,False,True,False,False
3,TCGA-D8-A1JN,620.0,80,13.0,1,True,True,False,False,False,...,False,False,True,False,False,True,False,True,False,False
4,TCGA-EW-A1P8,239.0,58,15.0,2,False,True,True,False,False,...,False,False,True,False,False,False,False,True,False,False


In [2]:
# 1. Afficher toutes les colonnes pour repérer l'âge et le stade
print("Liste des 26 colonnes cliniques :")
print(df_clinique.columns.tolist())

# 2. On trouve automatiquement la colonne qui contient le mot "vital"
colonne_survie = [col for col in df_clinique.columns if 'vital' in col.lower()][0]

# 3. Vérifier le déséquilibre des classes (Combien de 1 et combien de 2 ?)
print(f"\nRépartition des patients pour '{colonne_survie}' :")
print(df_clinique[colonne_survie].value_counts(dropna=False))

Liste des 26 colonnes cliniques :
['bcr_patient_barcode', 'Time', 'age_at_initial_pathologic_diagnosis', 'lymph_node_examined_count', 'vital_status', 'tissue_prospective_collection_indicator_YES', 'radiation_therapy_NO', 'breast_carcinoma_surgical_procedure_name_Lumpectomy', 'breast_carcinoma_surgical_procedure_name_Other', 'breast_carcinoma_surgical_procedure_name_Simple Mastectomy', 'menopause_status_Indeterminate (neither Pre or Postmenopausal)', 'menopause_status_Pre (<6 months since LMP AND no prior bilateral ovariectomy AND not on estrogen replacement)', 'pathologic_T_T1', 'pathologic_T_T3', 'pathologic_T_T4', 'pathologic_T_TX', 'pathologic_N_N1', 'pathologic_N_N2', 'pathologic_N_N3', 'pathologic_N_NX', 'pathologic_M_M1', 'pathologic_M_MX', 'pathologic_stage_Stage I', 'pathologic_stage_Stage III', 'pathologic_stage_Stage IV', 'pathologic_stage_Stage X']

Répartition des patients pour 'vital_status' :
vital_status
1    914
2    149
Name: count, dtype: int64


In [3]:
# 1. On liste tes colonnes sélectionnées (en gardant l'identifiant patient !)
colonnes_de_base = [
    'bcr_patient_barcode', 
    'Time', 
    'vital_status', 
    'age_at_initial_pathologic_diagnosis', 
    'lymph_node_examined_count', 
    'radiation_therapy_NO'
]

# On ajoute automatiquement toutes tes colonnes TNM et Stage (celles qui contiennent 'pathologic')
colonnes_tnm_stage = [col for col in df_clinique.columns if 'pathologic_' in col]
toutes_les_colonnes = colonnes_de_base + colonnes_tnm_stage

# 2. On crée notre nouveau tableau épuré
df_clean = df_clinique[toutes_les_colonnes].copy()
# Petite correction : on supprime les colonnes dupliquées (comme l'âge)
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()].copy()

# 3. On affiche la nouvelle taille et on compte les valeurs manquantes
print(f"Dimensions après retrait des doublons : {df_clean.shape}\n")
print("Compte des valeurs manquantes (NaN) par colonne :")
print(df_clean.isnull().sum())

Dimensions après retrait des doublons : (1063, 20)

Compte des valeurs manquantes (NaN) par colonne :
bcr_patient_barcode                    0
Time                                   0
vital_status                           0
age_at_initial_pathologic_diagnosis    0
lymph_node_examined_count              0
radiation_therapy_NO                   0
pathologic_T_T1                        0
pathologic_T_T3                        0
pathologic_T_T4                        0
pathologic_T_TX                        0
pathologic_N_N1                        0
pathologic_N_N2                        0
pathologic_N_N3                        0
pathologic_N_NX                        0
pathologic_M_M1                        0
pathologic_M_MX                        0
pathologic_stage_Stage I               0
pathologic_stage_Stage III             0
pathologic_stage_Stage IV              0
pathologic_stage_Stage X               0
dtype: int64


In [4]:
# 2. Exploration prudente du dossier des images
chemin_wsis = '/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/WSIs'

print("Aperçu des 10 premiers éléments dans le dossier WSIs :")
# os.scandir est ultra-rapide et ne fait pas planter la mémoire
with os.scandir(chemin_wsis) as entrees:
    for i, entree in enumerate(entrees):
        if i >= 10: # On s'arrête à 10
            break
        print(f" - {entree.name}")

Aperçu des 10 premiers éléments dans le dossier WSIs :


NameError: name 'os' is not defined

In [ ]:
# 1. On récupère le nom du premier dossier patient
premier_patient = next(os.scandir(chemin_wsis)).name
chemin_patient = os.path.join(chemin_wsis, premier_patient)

print(f"Dossier du patient analysé : {premier_patient}\n")
print("Aperçu des 5 premiers fichiers (patches) :")

# 2. On regarde les 5 premiers fichiers
with os.scandir(chemin_patient) as patches:
    for i, patch in enumerate(patches):
        if i >= 5: 
            break
        print(f" - {patch.name}")

# 3. On compte le nombre total de patches pour ce patient
nombre_patches = len(os.listdir(chemin_patient))
print(f"\nCe patient possède {nombre_patches} patches (tuiles d'images).")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# On reprend le chemin du patient précédent
chemin_patient = '/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/WSIs/TCGA-BH-A0C7'

# On liste tous ses fichiers et on prend les 4 premiers
fichiers = os.listdir(chemin_patient)[:4]

# On prépare un affichage : 1 ligne, 4 colonnes
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f"Patches histopathologiques du patient TCGA-BH-A0C7", fontsize=16)

# On boucle sur les 4 images pour les afficher
for i, fichier in enumerate(fichiers):
    chemin_complet = os.path.join(chemin_patient, fichier)
    
    # Ouverture et affichage de l'image
    img = Image.open(chemin_complet)
    axes[i].imshow(img)
    axes[i].set_title(f"Patch : {fichier}")
    axes[i].axis('off') # On cache les axes X et Y pour faire joli

plt.tight_layout()
plt.show()

In [ ]:
import os

# Le chemin exact vers le dataset de Sam Demharter
dossier_sam = '/kaggle/input/datasets/samdemharter'

print(f"--- Inventaire complet de {dossier_sam} ---")
for racine, _, fichiers in os.walk(dossier_sam):
    for fichier in fichiers:
        print(f"Dossier : {racine}")
        print(f"Fichier : {fichier}")

In [ ]:
import pandas as pd
import os

chemin_fichier = '/kaggle/input/datasets/samdemharter/brca-multiomics-tcga/brca_data_w_subtypes.csv'

# Ouverture
df_subtypes = pd.read_csv(chemin_fichier)

print(f"Dimensions du fichier : {df_subtypes.shape}")
print("\nListe des colonnes :")
print(df_subtypes.columns.tolist())

print("\n--- Aperçu des 5 premières lignes ---")
print(df_subtypes.head())